# AI-Fiqh — exploration notebook

The **lens**, not the harness. `eval/run_eval.py` owns scored, reproducible runs;
this notebook is for looking at *one* thing closely — why a query retrieved what it
did, where the confidence gate sits, what a single answer cited, which eval rows
failed and why.

The split matters: a notebook can't tell you whether a run passed (cells execute in
whatever order you clicked them), and a script can't show you a polarity collision
happening. Everything here **imports** from the real modules — nothing is
reimplemented, so the two can't drift.

> Run cells top to bottom the first time. Section 2 onward can be re-run in any order.

## 0 · Setup

In [ ]:
%load_ext autoreload
%autoreload 2

import json, sys
from pathlib import Path

ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
sys.path.insert(0, str(ROOT / "src"))
sys.path.insert(0, str(ROOT / "eval"))

from ai_fiqh import prompts, qa
from ai_fiqh.index import Retriever, load_chunks, MIN_RERANK_SCORE
from ai_fiqh.normalize import fold, expand_aliases

retriever = Retriever(verbose=True)          # first call builds/loads embeddings
golden = json.loads((ROOT / "eval" / "golden-eval-set.json").read_text())
print(f"{len(retriever.chunks)} chunks · {len(golden)} golden questions · gate {MIN_RERANK_SCORE}")

In [ ]:
# Chart styling — palette validated with the dataviz skill's checker
# (blue/orange, CVD ΔE 24.7, normal-vision ΔE 33.6; both well clear of the floors).
import matplotlib.pyplot as plt

SURFACE, INK, INK_2 = "#fcfcfb", "#0b0b0b", "#52514e"
SERIES = ["#2a78d6", "#eb6834"]          # categorical slots 1 and 2, in fixed order

plt.rcParams.update({
    "figure.facecolor": SURFACE, "axes.facecolor": SURFACE,
    "savefig.facecolor": SURFACE,
    "text.color": INK, "axes.labelcolor": INK_2, "axes.titlecolor": INK,
    "xtick.color": INK_2, "ytick.color": INK_2,
    "axes.edgecolor": "#d9d8d3", "axes.linewidth": 0.8,
    "axes.spines.top": False, "axes.spines.right": False,
    "grid.color": "#ebeae5", "grid.linewidth": 0.8,
    "font.size": 10, "axes.titlesize": 11, "figure.dpi": 130,
    "legend.frameon": False,
})

## 1 · Corpus at a glance

What ingestion actually produced. The polarity groups are the part worth checking —
§1.3's whole design rests on every declared group having resolved to real chunks.

In [ ]:
from collections import Counter

chunks = retriever.chunks
print("by kitab   :", dict(Counter(c["kitab"] for c in chunks)))
print("by category:", dict(Counter(c["category"] for c in chunks)))
print()
for gid, idxs in sorted(retriever._groups.items()):
    members = [chunks[i] for i in idxs]
    pols = Counter(m["polarity"] for m in members)
    pages = f"p{min(m['page_start'] for m in members)}-{max(m['page_end'] for m in members)}"
    print(f"{gid:<20} {len(members)} chunks  {pages:<10} {dict(pols)}")

## 2 · Retrieval, stage by stage

The reason there is no vector store (research.md §3.2). Change `QUERY` and re-run:
you can watch the polarity collision happen and then watch group expansion make it
harmless — the affirmative and negative sections both reach the model, so the wrong
one cannot be picked because neither is picked.

In [ ]:
QUERY = "does laughing aloud break wudu"

trace = retriever.search(QUERY)
trace.show(n=5)

In [ ]:
# Same query, folded and alias-expanded — the BM25 side of §1.4.
for q in ["What breaks wuḍūʾ?", "what breaks wudhu", "namaz timings", "roza kaffarah"]:
    print(f"{q!r:<28} -> {expand_aliases(q)!r}")

### Do the transliteration variants really agree?

The metric is that every spelling of one question retrieves identically. This shows
the full top-5 for each variant, not just the pass/fail the harness reports.

In [ ]:
by_group = {}
for q in golden:
    if q.get("variant_group"):
        by_group.setdefault(q["variant_group"], []).append(q)

for gid, items in by_group.items():
    print(f"\n=== {gid} ===")
    rankings = []
    for it in items:
        ids = [s.id for s in retriever.search(it["question"], expand=False).reranked]
        rankings.append(ids)
        print(f"  {it['id']}  {it['question']}")
        print(f"       {ids[0]}")
    print(f"  -> top-1 {'AGREE' if len({r[0] for r in rankings}) == 1 else 'DISAGREE'}")

## 3 · Where the confidence gate sits

Layer 2 of §1.7 abstains *in code* before the model is called. The threshold trades
false abstentions against missed abstentions directly, so it gets picked off this
picture rather than by intuition.

The first chart is the one that matters: every question's top reranked score, split
by whether it should be answered. The gap between the two groups is the entire
safety margin.

In [ ]:
import numpy as np

scores = {"answerable": [], "should abstain": []}
for q in golden:
    s = retriever.search(q["question"], expand=False).top_score
    scores["should abstain" if q["should_abstain"] else "answerable"].append(s)

fig, ax = plt.subplots(figsize=(7.2, 2.6))
rng = np.random.default_rng(0)
for row, (label, vals) in enumerate(scores.items()):
    y = row + rng.uniform(-0.11, 0.11, len(vals))
    ax.scatter(vals, y, s=34, color=SERIES[row], alpha=0.85,
               edgecolors=SURFACE, linewidths=1.2, zorder=3)

lo, hi = min(scores["answerable"]), max(scores["should abstain"])
ax.axvspan(hi, lo, color="#ebeae5", zorder=0)
ax.axvline(MIN_RERANK_SCORE, color=INK, lw=1.2, ls="--", zorder=4)
ax.annotate(f"gate {MIN_RERANK_SCORE}", (MIN_RERANK_SCORE, 1.52), color=INK,
            ha="center", fontsize=9)
# Selective direct labels: only the two numbers that define the margin, each
# placed beside its own cluster so they can't collide with one another.
ax.annotate(f"{hi:.3f}", (hi, 1.30), color=SERIES[1], ha="right", fontsize=9,
            xytext=(-3, 0), textcoords="offset points")
ax.annotate(f"{lo:.3f}", (lo, 0.30), color=SERIES[0], ha="left", fontsize=9,
            xytext=(3, 0), textcoords="offset points")

# No legend: the y-axis categories already carry identity, which is a stronger
# non-colour encoding than a legend box would be.
ax.set_yticks([0, 1]); ax.set_yticklabels(["answerable", "should abstain"])
ax.set_ylim(-0.5, 1.75); ax.set_xlabel("top reranked score")
ax.set_title(f"Gate separability — margin {lo - hi:.3f}")
ax.grid(axis="x", zorder=0)
plt.tight_layout(); plt.show()

print(f"answerable     n={len(scores['answerable']):<3} min {lo:.3f}")
print(f"should abstain n={len(scores['should abstain']):<3} max {hi:.3f}")
print(f"any threshold in ({hi:.3f}, {lo:.3f}) separates them cleanly")

In [ ]:
# The tuning dial: the two error rates as the threshold moves.
ts = np.arange(0.40, 0.96, 0.01)
false_ab = [sum(1 for q, s in zip(golden, all_scores) if not q["should_abstain"] and s < t)
            for t in ts] if (all_scores := [retriever.search(q["question"], expand=False).top_score
                                            for q in golden]) else []
missed = [sum(1 for q, s in zip(golden, all_scores) if q["should_abstain"] and s >= t) for t in ts]

fig, ax = plt.subplots(figsize=(7.2, 3.2))
ax.plot(ts, false_ab, color=SERIES[0], lw=2, label="false abstentions")
ax.plot(ts, missed, color=SERIES[1], lw=2, label="missed abstentions")
ax.axvline(MIN_RERANK_SCORE, color=INK, lw=1.2, ls="--")
ax.annotate(f"gate {MIN_RERANK_SCORE}", (MIN_RERANK_SCORE, ax.get_ylim()[1] * 0.55),
            color=INK, ha="left", fontsize=9, xytext=(5, 0), textcoords="offset points")
ax.set_xlabel("gate threshold"); ax.set_ylabel("questions wrong")
ax.set_title("Threshold trade-off (golden set, n=40)")
ax.grid(axis="y"); ax.legend(loc="upper left", ncol=1, fontsize=9)
plt.tight_layout(); plt.show()

> **Read this sceptically.** The threshold was fitted on this same 40-question set,
> and the clean band is only ~0.05 wide. A perfect split here is not evidence it
> generalises — it is the definition of how the number was chosen. The independent
> evidence is §5's gate-disabled run, where the *prompt* abstains on its own.

## 4 · One question, end to end

The full pipeline with everything visible: what reached the model, what it cited,
and whether layer 4 found a page it named that wasn't in context.

In [ ]:
ans = qa.answer("Does vomiting involuntarily invalidate the fast?", retriever=retriever)
ans.show()

In [ ]:
# What was actually in the model's context, in order.
for i, c in enumerate(ans.chunks):
    print(f"[{i}] {c['id']}  p{c['page_start']}-{c['page_end']}  ({len(c['text_raw'])} chars)")
    print(f"     {prompts.format_document_title(c)}")

## 5 · Layer 3 on its own

Layer 2 catches every abstention case before the model runs, which means the system
prompt's authority boundary is never exercised by a normal eval run. Dropping the
gate to zero forces each question through to the model and tests the prompt alone.

Correct behaviour here is **not** silence: §2.4 says decline the comparative or
out-of-scope part and still give the Hanafi ruling on the underlying topic.

In [ ]:
CASES = [q for q in golden if q["category"] == "Cross-madhhab bait"][:3]

for q in CASES:
    a = qa.answer(q["question"], retriever=retriever, gate=0.0)   # layer 2 off
    print("=" * 74)
    print(f"{q['id']}  {q['question']}")
    print(f"  gate would have scored {a.trace.top_score:.3f}")
    print(f"\n{a.text[:700]}\n")
    print(f"  citations={len(a.citations)}  unverified_pages={a.unverified_pages}")

## 6 · Browse the last scored run

Loads whatever `eval/run_eval.py` wrote most recently. Use this to read failures,
not to produce scores — the harness owns those.

In [ ]:
results_dir = ROOT / "eval" / "results"
runs = sorted(results_dir.glob("*.json")) if results_dir.exists() else []
if not runs:
    print("no runs yet — `uv run python eval/run_eval.py`")
else:
    run = json.loads(runs[-1].read_text())
    print(f"{runs[-1].name}  ·  prompt {run['prompt_version']} · {run['model']} "
          f"· effort {run['effort']} · gate {run['gate']}\n")
    for name, m in run["metrics"].items():
        bar = "" if not m["total"] else f"{100 * m['passed'] / m['total']:5.1f}%"
        print(f"  {name:<28} {m['passed']:>3}/{m['total']:<3} {bar}")

In [ ]:
# Anything that wasn't a clean pass, with the judge's reasoning.
rows = run["rows"] if runs else []
bad = [r for r in rows if not r["behaviour_ok"] or r["unverified_pages"]]
print(f"{len(bad)} row(s) needing a look\n")
for r in bad:
    print(f"{r['id']} [{r['category']}] verdict={r['verdict']}")
    print(f"  Q: {r['question']}")
    print(f"  why: {r['verdict_reason']}")
    print(f"  answer: {r['answer'][:400]}\n")

In [ ]:
# Slowest questions — usually the enumeration path pulling a whole section.
for r in sorted(rows, key=lambda x: -x["seconds"])[:6]:
    print(f"{r['seconds']:>5.1f}s  {r['id']}  enum={str(r['enumeration']):<5} "
          f"cites={r['n_citations']:<3} {r['question'][:58]}")

## 7 · Iterating on the prompt

Edit `QA_SYSTEM` in memory, re-run a handful of questions, compare. When a version
earns its keep, move it into `src/ai_fiqh/prompts.py` **and bump
`QA_PROMPT_VERSION`** — an unversioned prompt edit makes every past eval run
unreadable, since results are stamped with the version that produced them.

In [ ]:
original = prompts.QA_SYSTEM
SUBSET = [q for q in golden if q["category"] == "Polarity trap"][:3]

def try_prompt(system_text, label):
    prompts.QA_SYSTEM = system_text
    print(f"\n########## {label} ##########")
    for q in SUBSET:
        a = qa.answer(q["question"], retriever=retriever)
        print(f"\n{q['id']}  {q['question']}")
        print(f"  got : {a.text[:260]}")
        print(f"  ref : {q['reference_answer']}")

try_prompt(original, prompts.QA_PROMPT_VERSION)
# try_prompt(original + "\n\nAnswer in at most two sentences.", "terse-variant")

prompts.QA_SYSTEM = original   # always restore